# Encodage des personas : Tate McRae (Tatiana) et Beyoncé (Sasha Fierce)

| Artiste | Baseline | Persona |
| :---- | :---- | :---- |
| Tate McRae | `persona_tate_baseline` | `persona_tatiana` |
| Beyoncé | `persona_beyonce_baseline` | `persona_sasha_fierce` |

**Étapes :** téléchargement Genius → nettoyage → formatage JSON → normalisation → attribution des personas → fiches persona → export dans `Corpus/` → contrôles.

**Prérequis :** dépôt à jour (`git pull`), token Genius dans `~/GENIUS_TOKEN.txt` (hors du dépôt). Le notebook ne modifie aucun script partagé.

**Fichiers produits :**
- `travail/<artiste>/` : fichiers intermédiaires (téléchargements bruts). Non poussés.
- `Corpus/TateMcRae/` et `Corpus/Beyonce/` : `<Nom>_songs/` (un JSON par chanson, `persona_id` sur chaque ligne), `<Nom>_personas/` (fiches persona), `<Nom>_attribution_personas.csv` (attribution, score de confiance, source).
- `Corpus/personas/<Nom>_personas/` : copie des fiches persona.

Le notebook peut être relancé de haut en bas : chaque étape repart de fichiers propres.

## 0. Configuration

In [ ]:
import ast, glob, json, os, re, shutil, unicodedata
import pandas as pd
import requests

import data_api_by_artist as genius
import formatting

with open(os.path.join(os.path.expanduser("~"), "GENIUS_TOKEN.txt"), encoding="utf-8") as f:
    ACCESS_TOKEN = f.read().strip()

TRAVAIL = "travail"                                   # fichiers intermédiaires, non poussés
ANCIEN = os.path.join("..", "PardonMyFrench_OLD", "corpus")   # anciens téléchargements, réutilisés s'ils suffisent
NOMS = {"tate_mcrae": "TateMcRae", "beyonce": "Beyonce"}

# Genius ajoute parfois une ligne « [Produced by ...] » en tête des paroles :
# on l'ignore ici, sans modifier formatting.py, pour qu'elle ne devienne pas une fausse section.
formatting.IGNORED = re.compile(r"(?:Paroles de|Produced by)\b.*", re.I)

## 1. Choix des albums

Écartés : remixes, lives, sessions, singles physiques, versions « Original Tracklist » / « Physical Release » / « Digital Deluxe » (la deluxe `1489439` contient déjà les 21 titres de *So Close to What*), *TM4\** (non sorti).

*4* (2011) est ajouté comme baseline de Beyoncé : premier album après l'annonce de la « mort » de Sasha Fierce (2010). Il rééquilibre le volume de texte et contient aussi des titres uptempo, ce qui évite que la baseline ne soit faite que de ballades.

In [ ]:
ALBUMS = {
    "tate_mcrae": [
        547004,    # all the things i never said (EP, 2020)   → baseline
        669743,    # TOO YOUNG TO BE SAD (EP, 2021)           → baseline
        796536,    # i used to think i could fly (2022)       → baseline
        963023,    # THINK LATER (2023)                       → Tatiana
        1489439,   # SO CLOSE TO WHAT??? (deluxe, 2025)       → Tatiana (en partie)
    ],
    "beyonce": [
        610619,    # I Am... Sasha Fierce (Deluxe Edition) : disque « I Am... » + disque « Sasha Fierce »
        32041,     # 4 (2011) : après la « mort » de Sasha Fierce → baseline
    ],
}

## 2. Téléchargement des paroles (Genius)

Pour chaque artiste, on réutilise un téléchargement existant s'il contient déjà tous les albums demandés (dans `travail/`, sinon dans l'ancienne copie du dépôt). Sinon, on télécharge (plusieurs minutes). Le fichier brut `tracksDf_brut.csv` n'est jamais modifié ensuite.

In [ ]:
def couvre(chemin, ids):
    return os.path.exists(chemin) and set(ids) <= set(pd.read_csv(chemin)["album_id"])

for artiste, ids in ALBUMS.items():
    dossier = os.path.join(TRAVAIL, artiste)
    os.makedirs(dossier, exist_ok=True)
    brut = os.path.join(dossier, "tracksDf_brut.csv")
    ancien = os.path.join(ANCIEN, artiste, "tracksDf_avant_nettoyage.csv")
    if couvre(brut, ids):
        print(f"{artiste} : téléchargement déjà présent")
    elif couvre(ancien, ids):
        shutil.copy(ancien, brut)
        print(f"{artiste} : repris de l'ancienne copie du dépôt")
    else:
        print(f"{artiste} : téléchargement depuis Genius…")
        df = pd.concat([genius.get_album_tracks(a, access_token=ACCESS_TOKEN) for a in ids],
                       ignore_index=True)
        df.to_csv(brut, index=False)
    df = pd.read_csv(brut)
    df = df[df["album_id"].isin(ids)]
    print(f"   {len(df)} titres : " + ", ".join(f"{a} ({(df['album_id'] == a).sum()})" for a in ids))

## 3. Nettoyage

Retraits automatiques, chacun affiché avec sa raison :
- **featurings** : le code ne sait pas séparer les voix, une partie des paroles serait celle de l'invité ;
- **versions dérivées** (remix, live, acoustique, instrumental…) : doublons ;
- **textes trop courts** (< 50 mots) : interludes ;
- **doublons de titre**.

Plus une liste de retraits manuels justifiés.

In [ ]:
RETRAITS_MANUELS = {
    "tate_mcrae": {
        11469778: "bloodonmyhands : featuring probable de Flo Milli (créditée comme parolière)",
    },
    "beyonce": {
        85389: "Poison : piste bonus hors des deux disques, pas d'attribution possible",
    },
}
DERIVE = re.compile(r"\(.*\b(remix|live|acoustic|instrumental|version|edit|mix)\b.*\)", re.I)

for artiste, ids in ALBUMS.items():
    df = pd.read_csv(os.path.join(TRAVAIL, artiste, "tracksDf_brut.csv"))
    df = df[df["album_id"].isin(ids)].copy()
    df["language"] = "en"   # la langue détectée par Genius n'est pas fiable
    df["nb_mots"] = df["lyrics"].fillna("").str.split().str.len()
    raisons = {}
    for r in df.itertuples():
        feats = ast.literal_eval(r.featured_artists_names) if isinstance(r.featured_artists_names, str) else []
        if r.id in RETRAITS_MANUELS.get(artiste, {}):
            raisons[r.id] = RETRAITS_MANUELS[artiste][r.id]
        elif feats:
            raisons[r.id] = f"featuring : {', '.join(feats)}"
        elif DERIVE.search(r.title):
            raisons[r.id] = "version dérivée"
        elif r.nb_mots < 50:
            raisons[r.id] = f"texte trop court ({r.nb_mots} mots)"
    df = df[~df["id"].isin(raisons)]
    doublons = df[df.duplicated("title", keep="first")]
    for r in doublons.itertuples():
        raisons[r.id] = "doublon de titre"
    df = df.drop(doublons.index)
    df.drop(columns="nb_mots").to_csv(os.path.join(TRAVAIL, artiste, "tracksDf.csv"), index=False)
    print(f"\n=== {artiste} : {len(df)} chansons gardées, {len(raisons)} retirées ===")
    brut = pd.read_csv(os.path.join(TRAVAIL, artiste, "tracksDf_brut.csv")).set_index("id")["title"]
    for i, raison in raisons.items():
        print(f"  - {brut[i]} ({i}) : {raison}")

## 4. Formatage en JSON (un fichier par chanson)

Le dossier `songs/` est supprimé puis recréé à chaque exécution.

In [ ]:
for artiste in ALBUMS:
    base = os.path.join(TRAVAIL, artiste)
    shutil.rmtree(os.path.join(base, "songs"), ignore_errors=True)
    df = pd.read_csv(os.path.join(base, "tracksDf.csv"))
    formatting.bunch_of_songs_formatter(df=df, output_folder=os.path.join(base, "songs"))

## 5. Normalisation

- **Filigrane Genius :** des lettres cyrilliques sosies (« е » pour « e »…) sont insérées dans certaines paroles. Elles fausseraient fréquences de mots et n-grammes de caractères : on les remplace par les lettres latines.
- **Paroliers :** `lyricists` sort du CSV sous forme de texte (`"[1, 2]"`) ; on le convertit en vraie liste.

In [ ]:
HOMOGLYPHES = str.maketrans("аеорсухіјАЕОРСХТНКМВ", "aeopcyxijAEOPCXTHKMB")

for artiste in ALBUMS:
    n_car = 0
    fichiers = glob.glob(os.path.join(TRAVAIL, artiste, "songs", "*.json"))
    for path in fichiers:
        with open(path, encoding="utf-8") as f:
            song = json.load(f)
        for section in song["lyrics"]:
            for line in section["lines"]:
                avant = line["text"]
                line["text"] = avant.translate(HOMOGLYPHES)
                n_car += sum(a != b for a, b in zip(avant, line["text"]))
        song["title"] = song["title"].translate(HOMOGLYPHES)
        if isinstance(song["lyricists"], str):
            song["lyricists"] = ast.literal_eval(song["lyricists"])
        with open(path, "w", encoding="utf-8") as f:
            json.dump(song, f, ensure_ascii=False, indent=2)
    print(f"{artiste} : {len(fichiers)} fichiers, {n_car} caractères cyrilliques corrigés")

## 6. Règles d'attribution des personas

Le score de confiance reflète **la solidité de la source**, pas une impression sur la chanson.

**Beyoncé**
- Disque « I Am... » → baseline (1.0)
- Disque « Sasha Fierce » → Sasha Fierce (1.0), sauf *Hello*, *Ego*, *Scared of Lonely* (0.7) : musicalement Sasha, textuellement vulnérables
- Album *4* (2011) → baseline (0.8) : après la « mort » de Sasha Fierce ; l'artiste parle de fusion des deux facettes plutôt que d'un retour à une baseline pure

**Tate McRae**
- EP et *i used to think i could fly* → baseline (1.0) : antérieurs à l'apparition de Tatiana
- *greedy*, *It's ok I'm ok* → Tatiana (1.0) : attribués par l'artiste
- reste de *Think Later* → Tatiana (0.8) : album dit inspiré par Tatiana
- *Sports car*, *Revolving door*, *Tit for Tat* → Tatiana (0.7) : association faite par la presse
- reste de *So Close to What* → Tatiana (0.5) : Tatiana en aurait écrit « peut-être la moitié »

In [ ]:
# ---------- Beyoncé ----------
DISQUE_1 = [83613, 52158, 78577, 52683, 55885, 4948, 75936, 152215]
DISQUE_2 = [50873, 152213, 68973, 56878, 1829007, 80249, 152217, 152218]
HYBRIDES = {
    80249:  "ballade au piano placée sur le disque Sasha Fierce",
    152217: "musicalement Sasha, textuellement vulnérable (MTV News)",
    152218: "musicalement Sasha, textuellement vulnérable (MTV News)",
}
ALBUM_4 = 32041

def attribuer_beyonce(r):
    if r.id in DISQUE_1:
        return "persona_beyonce_baseline", 1.0, "disque 1 « I Am... » de l'édition deluxe (étiquetage de l'artiste)"
    if r.id in HYBRIDES:
        return "persona_sasha_fierce", 0.7, "disque 2 « Sasha Fierce », mais " + HYBRIDES[r.id]
    if r.id in DISQUE_2:
        return "persona_sasha_fierce", 1.0, "disque 2 « Sasha Fierce » de l'édition deluxe (étiquetage de l'artiste)"
    if r.album_id == ALBUM_4:
        return "persona_beyonce_baseline", 0.8, "album 4 (2011), après l'annonce de la « mort » de Sasha Fierce (2010) : fusion revendiquée par l'artiste"
    return None, None, "hors des règles : non attribuée"

# ---------- Tate McRae ----------
ALBUMS_BASELINE = [547004, 669743, 796536]
THINK_LATER = 963023
GREEDY, ITS_OK_IM_OK = 9393794, 10794283
SINGLES_PRESSE = [11103036, 10955668, 12456608]   # Sports car, Revolving door, Tit for Tat

def attribuer_tate(r):
    if r.album_id in ALBUMS_BASELINE:
        return "persona_tate_baseline", 1.0, "sortie antérieure à l'apparition de Tatiana en tournée"
    if r.id == GREEDY:
        return "persona_tatiana", 1.0, "greedy explicitement attribué à Tatiana par l'artiste (Billboard)"
    if r.album_id == THINK_LATER:
        return "persona_tatiana", 0.8, "album Think Later dit inspiré par Tatiana par l'artiste (Billboard)"
    if r.id == ITS_OK_IM_OK:
        return "persona_tatiana", 1.0, "attribué à Tatiana par l'artiste (Ryan Seacrest)"
    if r.id in SINGLES_PRESSE:
        return "persona_tatiana", 0.7, "single associé à Tatiana par la presse, pas par l'artiste"
    return "persona_tatiana", 0.5, "l'artiste dit que Tatiana a écrit « peut-être la moitié » de l'album : incertain"

REGLES = {"beyonce": attribuer_beyonce, "tate_mcrae": attribuer_tate}

for artiste, regle in REGLES.items():
    att = pd.read_csv(os.path.join(TRAVAIL, artiste, "tracksDf.csv"))[["id", "title", "album_id"]]
    att[["persona_id", "confidence_score", "source"]] = [regle(r) for r in att.itertuples()]
    att.to_csv(os.path.join(TRAVAIL, artiste, "attribution_personas.csv"), index=False, encoding="utf-8")
    print(f"{artiste} : {att['persona_id'].notna().sum()} personas attribuées sur {len(att)}")
    if att["persona_id"].isna().any():
        print("   non attribuées :", ", ".join(att.loc[att["persona_id"].isna(), "title"]))

## 7. Application des personas aux JSON

Conformément à `schemaChanson.json`, chaque ligne reçoit un champ `persona_id`. Le score de confiance reste dans le CSV d'attribution.

In [ ]:
for artiste in ALBUMS:
    att = pd.read_csv(os.path.join(TRAVAIL, artiste, "attribution_personas.csv")).dropna(subset=["persona_id"])
    att = dict(zip(att["id"].astype(str), att["persona_id"]))
    n = 0
    for path in glob.glob(os.path.join(TRAVAIL, artiste, "songs", "*.json")):
        with open(path, encoding="utf-8") as f:
            song = json.load(f)
        pid = att.get(str(song["song_id"]))
        if pid:
            for section in song["lyrics"]:
                for line in section["lines"]:
                    line["persona_id"] = pid
            n += 1
        with open(path, "w", encoding="utf-8") as f:
            json.dump(song, f, ensure_ascii=False, indent=2)
    print(f"{artiste} : persona ajoutée à {n} chansons")

## 8. Fiches persona

Un JSON par persona (format `schemaPersona.json`), avec les identifiants Genius des artistes et albums.

Périodes datées par les sorties. Tatiana commence avec *Think Later* (2023). Sasha Fierce s'arrête en 2010, quand Beyoncé annonce l'avoir « tuée ». La baseline de Beyoncé s'appuie sur *I Am... Sasha Fierce* (disque « I Am... ») et sur *4*.

In [ ]:
def artist_id(requete):
    r = requests.get("https://api.genius.com/search", params={"q": requete},
                     headers={"Authorization": f"Bearer {ACCESS_TOKEN}"})
    res = r.json()["response"]["hits"][0]["result"]["primary_artist"]
    print(f"{res['name']} → id {res['id']}")
    return str(res["id"])

TATE = artist_id("Tate McRae greedy")
BEY = artist_id("Beyoncé Halo")

PERSONAS = {
    "tate_mcrae": [
        {"persona_id": "persona_tate_baseline", "persona_name": "Tate McRae", "artist_id": TATE,
         "period": {"start": "2020", "end": "2022"},
         "anchored_releases": ["547004", "669743", "796536"], "is_baseline": True},
        {"persona_id": "persona_tatiana", "persona_name": "Tatiana", "artist_id": TATE,
         "period": {"start": "2023", "end": None},
         "anchored_releases": ["963023", "1489439"], "is_baseline": False},
    ],
    "beyonce": [
        {"persona_id": "persona_beyonce_baseline", "persona_name": "Beyoncé", "artist_id": BEY,
         "period": {"start": "2008", "end": None},
         "anchored_releases": ["610619", "32041"], "is_baseline": True},
        {"persona_id": "persona_sasha_fierce", "persona_name": "Sasha Fierce", "artist_id": BEY,
         "period": {"start": "2008", "end": "2010"},
         "anchored_releases": ["610619"], "is_baseline": False},
    ],
}

for artiste, liste in PERSONAS.items():
    dossier = os.path.join(TRAVAIL, artiste, "personas")
    shutil.rmtree(dossier, ignore_errors=True)
    os.makedirs(dossier)
    for p in liste:
        with open(os.path.join(dossier, f"{p['persona_id']}.json"), "w", encoding="utf-8") as f:
            json.dump(p, f, ensure_ascii=False, indent=2)
        print("Créé :", p["persona_id"])

## 9. Export vers `Corpus/`

Selon l'organisation actuelle du dépôt : `Corpus/<Nom>/<Nom>_songs/`, `<Nom>_personas/`, `<Nom>_attribution_personas.csv`, plus une copie des fiches dans `Corpus/personas/<Nom>_personas/`. Les anciens dossiers sont remplacés (ce qui retire par exemple *Poison* s'il y était encore). Le champ `album_id` déjà présent dans le dépôt est conservé.

In [ ]:
for artiste, nom in NOMS.items():
    src = os.path.join(TRAVAIL, artiste)
    base = os.path.join("Corpus", nom)
    os.makedirs(base, exist_ok=True)
    # album_id : champ ajouté par l'équipe dans chaque chanson. On garde la valeur déjà présente
    # dans le dépôt ; pour une nouvelle chanson, on prend l'album d'où elle a été téléchargée.
    dossier_songs = os.path.join(base, f"{nom}_songs")
    album_ids = dict(zip(pd.read_csv(os.path.join(src, "tracksDf.csv"))["id"].astype(str),
                         pd.read_csv(os.path.join(src, "tracksDf.csv"))["album_id"].astype(int)))
    for path in glob.glob(os.path.join(dossier_songs, "*.json")):
        with open(path, encoding="utf-8") as f:
            ancien = json.load(f)
        if "album_id" in ancien:
            album_ids[str(ancien["song_id"])] = ancien["album_id"]
    cibles = {
        os.path.join(base, f"{nom}_songs"): os.path.join(src, "songs"),
        os.path.join(base, f"{nom}_personas"): os.path.join(src, "personas"),
        os.path.join("Corpus", "personas", f"{nom}_personas"): os.path.join(src, "personas"),
    }
    for dest, origine in cibles.items():
        shutil.rmtree(dest, ignore_errors=True)
        shutil.copytree(origine, dest)
    for path in glob.glob(os.path.join(dossier_songs, "*.json")):
        with open(path, encoding="utf-8") as f:
            song = json.load(f)
        song["album_id"] = album_ids[str(song["song_id"])]
        with open(path, "w", encoding="utf-8") as f:
            json.dump(song, f, ensure_ascii=False, indent=2)
    shutil.copy(os.path.join(src, "attribution_personas.csv"),
                os.path.join(base, f"{nom}_attribution_personas.csv"))
    print(f"{artiste} → Corpus/{nom}/ et Corpus/personas/{nom}_personas/")

## 10. Contrôles

Attendu : aucun JSON vide, un `album_id` dans chaque chanson, aucune lettre cyrillique, aucune section « produced », une seule persona par chanson. Le tableau donne le volume de texte par persona (mots, hors adlibs entre parenthèses).

In [ ]:
for artiste, nom in NOMS.items():
    fichiers = glob.glob(os.path.join("Corpus", nom, f"{nom}_songs", "*.json"))
    vides, cyr, produced, lignes = [], 0, 0, []
    for path in fichiers:
        with open(path, encoding="utf-8") as f:
            song = json.load(f)
        if not song["lyrics"]:
            vides.append(song["title"])
        produced += sum(s["type"] == "produced" for s in song["lyrics"])
        ls = [l for s in song["lyrics"] for l in s["lines"]]
        cyr += sum("CYRILLIC" in unicodedata.name(c, "") for l in ls for c in l["text"])
        if "album_id" not in song:
            vides.append(song["title"] + " (sans album_id)")
        pids = {l.get("persona_id") for l in ls}
        mots = sum(len(l["text"].split()) for l in ls)
        mots_sans_adlibs = sum(len(re.sub(r"\([^)]*\)", "", l["text"]).split()) for l in ls)
        lignes.append({"persona": pids.pop() if len(pids) == 1 else "MIXTE",
                       "chansons": 1, "lignes": len(ls), "mots": mots, "mots_sans_adlibs": mots_sans_adlibs})
    t = pd.DataFrame(lignes).groupby("persona", dropna=False).sum()
    t["part_des_mots"] = (t["mots"] / t["mots"].sum() * 100).round().astype(int).astype(str) + " %"
    print(f"\n=== {nom} : {len(fichiers)} chansons ===")
    print(t.to_string())
    print("Paroles vides :", vides or "aucune", "| lettres cyrilliques :", cyr, "| sections « produced » :", produced)